In [1]:
from collections import Counter  
import numpy as np

### Process a text file containing a sequence of a subregion of the annotated genome to build HMM.
**Chromosome range:** chr12:56,741,761-56,761,390

In [2]:
def pull_data(path):
    raw = "".join(line.strip() for line in open(path) if not line.startswith(">"))
    sequence = [b.upper() for b in raw]
    hidden_states = ["CpG" if b.isupper() else "non-CpG" for b in raw] 
    return sequence, hidden_states 

In [3]:
sequence, hidden_states = pull_data("mm39ExtendedDNAOutput.txt")

In [4]:
print(len(sequence))

19630


In [5]:
print(hidden_states.count("CpG"))

787


In [6]:
print(hidden_states.count("non-CpG"))

18843


### 1. Getting the Transition Probability:
Determine the number of different transitions

## I might have to change the states and probabilitese to optimize performace once i have the viterbi algorithmn finished.

In [7]:
transition = {}
for current_state, next_state in zip(hidden_states[:-1], hidden_states[1:]):
    # initialize count for all possible transitions
    if (current_state, next_state) not in transition:
        transition[(current_state, next_state)] = 1 

    else: 
        transition[(current_state, next_state)] += 1

In [8]:
print (transition)

{('non-CpG', 'non-CpG'): 18841, ('non-CpG', 'CpG'): 1, ('CpG', 'CpG'): 786, ('CpG', 'non-CpG'): 1}


In [9]:
transition_probability = {}

for (state1, state2), count in transition.items():
    
    total = sum( c for (s1, s2), c in transition.items() if s1==state1 ) 
    if state1 not in transition_probability:
    
        transition_probability[state1] = {}
    transition_probability[state1][state2] = count / total

In [10]:
print(transition_probability)

{'non-CpG': {'non-CpG': 0.9999469270778049, 'CpG': 5.307292219509606e-05}, 'CpG': {'CpG': 0.9987293519695044, 'non-CpG': 0.0012706480304955528}}


### 2. Getting the Emission Probability:

In [11]:
counts = {"CpG": Counter(), "non-CpG": Counter()}

for base, state in zip(sequence, hidden_states):
    counts[state][base] += 1 

print(counts)

{'CpG': Counter({'C': 257, 'G': 237, 'A': 148, 'T': 145}), 'non-CpG': Counter({'T': 5673, 'A': 5497, 'G': 3855, 'C': 3818})}


In [12]:
emission = {}

for state, count in counts.items():
    total = sum(count.values())
    emission[state] = [round(count["A"] / total, 4), 
                        round(count["T"] / total, 4),
                        round(count["C"] / total, 4),
                        round(count["G"] / total, 4),] 

for state in emission:
    print(state, emission[state])

CpG [0.1881, 0.1842, 0.3266, 0.3011]
non-CpG [0.2917, 0.3011, 0.2026, 0.2046]


In [13]:
emission_probability = { 
    "CpG": {"A": 0.1881, "T": 0.1842, "C": 0.3266, "G": 0.3011},
    "non-CpG": {"A": 0.2917, "T": 0.3011, "C": 0.2026, "G": 0.2046}
}

### 3. Implementing the Viterbi Algorithm:
<br>
The Viterbi algorithm is implemented to infer the most probable hidden states based on an obserable

In [14]:
def viterbi(seq, states, transition_prob, emission_prob, initial_prob):
    
    num_row    = len(states)
    num_column = len(seq) 

    # Initialize a matrix to track the maximum probability of reaching a hidden state at each position
    bestpathprob = np.zeros((num_row, num_column))

    # Initialize a matrix to track...
    bestpathpointer = np.zeros((num_row, num_column), dtype=int)

    # Step 1: Initialization 
    for idx, state in enumerate(states):
        bestpathprob[idx, 0] = initial_prob[state] * emission_prob[state][seq[0]]

    # We start iterating over the remaining sequences (start at index 1 since we have initialized
    # index 0 at step 1 
    for t in range(1, len(seq)):
        for idx, curr_state in enumerate(states): 
            
            prob_history = []

            for i, prev_state in enumerate(states):
                curr_prob = transition_prob[prev_state][curr_state] * emission_prob[curr_state][seq[t]]
                prev_prob = bestpathprob[i, t-1]
                
                prob_history.append(curr_prob * prev_prob)
                
            bestpathprob[idx, t] = max(prob_history)

            # np.argmax will return the index of the maximum value in the NumPy array.
            # We can then use the index to find the state.
            bestpathpointer[idx, t] = np.argmax(prob_history)
                            

    # Termination steps: 
    # Find the maximum probability from the final stage
    bestprob_last = np.argmax(bestpathprob[:, -1])
    bestprob = np.max(bestpathprob[:, -1])


    # Reconstruct the path leading to the final state
    path = [states[bestprob_last]] 
    for i in range(num_column-1, 0, -1): 

        backpointer = bestpathpointer[bestprob_last, i]
        path.append(states[backpointer])

        bestprob_last = backpointer
       
         
    bestpath = path[::-1]
    

    return bestpath, bestprob

In [15]:
initial_probability = {

    "CpG": 0,

    "non-CpG": 1,

}

states = ["CpG", "non-CpG"]

bestpath, prob = viterbi(sequence, states, transition_probability, emission_probability, initial_probability)



In [16]:
bestpath.count("CpG")

19134